# German Credit 评分卡演示\n
\n
本 Notebook 演示完整流程：\n
1. 加载 germancredit 数据\n
2. 分箱与 WOE 转换\n
3. 训练 ScoreCard（descending/ascending）\n
4. 校验分数方向与概率关系\n
5. 查看 score_odds_reference 基准分参照\n

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Ensure local hscredit import works from notebook context
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from hscredit.core.binning import OptimalBinning
from hscredit.core.models import ScoreCard
from hscredit.utils.datasets import germancredit

c:\Users\18306\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\18306\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def summarize_direction_result(name: str, y_true: pd.Series, scores: np.ndarray, proba: np.ndarray) -> None:
    good_mean = float(scores[y_true == 0].mean())
    bad_mean = float(scores[y_true == 1].mean())
    corr = float(np.corrcoef(scores, proba)[0, 1])

    print(f'[{name}]')
    print(f'score range       : [{scores.min():.2f}, {scores.max():.2f}]')
    print(f'mean score good=0 : {good_mean:.2f}')
    print(f'mean score bad=1  : {bad_mean:.2f}')
    print(f'corr(score, pbad) : {corr:.4f}')

    if name == 'descending':
        print(f'expect good > bad : {good_mean > bad_mean}')
        print(f'expect corr < 0   : {corr < 0}')
    else:
        print(f'expect bad > good : {bad_mean > good_mean}')
        print(f'expect corr > 0   : {corr > 0}')

## 1) 加载数据

In [3]:
df = germancredit().copy()
target_col = 'class'
y = df[target_col].astype(int)
X = df.drop(columns=[target_col])

print('German Credit loaded')
print(f'shape             : {df.shape}')
print(f'features          : {X.shape[1]}')
print(f'bad rate (class=1): {y.mean():.4f}')
df.head()

German Credit loaded
shape             : (1000, 21)
features          : 20
bad rate (class=1): 0.3000


,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_and_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_or_guarantors,...,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker,class
0,... < 0 DM,6,critical account/ other credits existing (not ...,radio/television,1169,unknown/ no savings account,... >= 7 years,4,male : divorced/separated,none,...,real estate,67,none,own,2,skilled employee / official,1,"yes, registered under the customers name",yes,0
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,radio/television,5951,... < 100 DM,1 <= ... < 4 years,2,male : divorced/separated,none,...,real estate,22,none,own,1,skilled employee / official,1,none,yes,1
2,no checking account,12,critical account/ other credits existing (not ...,education,2096,... < 100 DM,4 <= ... < 7 years,2,male : divorced/separated,none,...,real estate,49,none,own,1,unskilled - resident,2,none,yes,0
3,... < 0 DM,42,existing credits paid back duly till now,furniture/equipment,7882,... < 100 DM,4 <= ... < 7 years,2,male : divorced/separated,guarantor,...,building society savings agreement/ life insur...,45,none,for free,1,skilled employee / official,2,none,yes,0
4,... < 0 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : divorced/separated,none,...,unknown / no property,53,none,for free,2,skilled employee / official,2,none,yes,1


## 2) 划分训练测试 + 分箱与 WOE

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

binner = OptimalBinning(method='target_bad_rate', max_n_bins=5)
binner.fit(X_train, y_train)

X_train_woe = binner.transform(X_train, metric='woe')
X_test_woe = binner.transform(X_test, metric='woe')

print('X_train_woe shape:', X_train_woe.shape)
X_train_woe.head()

X_train_woe shape: (700, 20)


,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_and_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_or_guarantors,present_residence_since,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker
10,0.415165,-0.344096,0.082842,0.173721,-0.043939,0.301787,0.579034,0.100083,-0.287682,-0.005997,0.071459,0.034804,0.455256,-0.174622,0.468861,-0.004193,-0.039556,-0.005666,0.044150,0.044951
82,-1.184134,0.101228,0.082842,0.173721,-0.043939,0.098061,-0.074691,0.100083,-0.114603,-0.005997,-0.010989,-0.051449,0.455256,-0.174622,0.468861,-0.004193,-0.104261,-0.005666,0.044150,0.044951
827,-1.184134,0.101228,1.134980,0.173721,-0.043939,0.301787,-0.074691,-0.211784,0.069593,-0.005997,-0.010989,0.034804,-0.264053,0.308301,-0.214468,-0.004193,-0.039556,0.029853,0.044150,0.044951
410,0.415165,0.101228,0.082842,-0.384846,-0.043939,0.301787,-0.194156,0.149489,0.069593,-0.005997,-0.010989,0.034804,0.455256,-0.174622,-0.214468,-0.004193,-0.039556,-0.005666,-0.066521,0.044951
48,-1.184134,-0.344096,-0.699082,0.173721,-0.043939,0.301787,-0.074691,-0.292136,-0.287682,-0.005997,-0.010989,-0.051449,-0.264053,-0.174622,-0.214468,-0.004193,-0.104261,-0.005666,0.044150,0.044951


## 3) 信用分模式（descending）

In [5]:
scorecard_desc = ScoreCard(
    pdo=60,
    rate=2,
    base_odds=35,
    base_score=750,
    direction='descending',
    binner=binner,
)
scorecard_desc.fit(X_train_woe, y_train, input_type='woe')

proba_desc = scorecard_desc.predict_proba(X_test)[:, 1]
score_desc = scorecard_desc.predict(X_test, input_type='raw')

auc_desc = roc_auc_score(y_test, proba_desc)
print(f'AUC (descending): {auc_desc:.4f}')
summarize_direction_result('descending', y_test, score_desc, proba_desc)

AUC (descending): 0.8028
[descending]
score range       : [238.34, 2287.31]
mean score good=0 : 609.38
mean score bad=1  : 452.91
corr(score, pbad) : -0.6518
expect good > bad : True
expect corr < 0   : True


## 4) 欺诈分模式（ascending）

In [6]:
scorecard_asc = ScoreCard(
    pdo=60,
    rate=2,
    base_odds=35,
    base_score=750,
    direction='ascending',
    binner=binner,
)
scorecard_asc.fit(X_train_woe, y_train, input_type='woe')

proba_asc = scorecard_asc.predict_proba(X_test)[:, 1]
score_asc = scorecard_asc.predict(X_test, input_type='raw')

auc_asc = roc_auc_score(y_test, proba_asc)
print(f'AUC (ascending): {auc_asc:.4f}')
summarize_direction_result('ascending', y_test, score_asc, proba_asc)

AUC (ascending): 0.8051
[ascending]
score range       : [-755.53, 1259.77]
mean score good=0 : 891.82
mean score bad=1  : 1048.39
corr(score, pbad) : 0.6594
expect bad > good : True
expect corr > 0   : True


In [7]:
scorecard_asc.scorecard_points()

,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,515.35,NaN
1,status_of_existing_checking_account,,"[-inf, 0.5)",-51.46,0.7639
2,status_of_existing_checking_account,,"[0.5, 1.5)",8.99,-0.1335
3,status_of_existing_checking_account,,"[1.5, 2.5)",-27.97,0.4152
4,status_of_existing_checking_account,,"[2.5, +inf)",79.76,-1.1841
...,...,...,...,...,...
74,number_of_people_being_liable_to_provide_maint...,,"[1.5, +inf)",-0.18,0.0299
75,telephone,,"[-inf, 0.5)",-1.89,0.0442
76,telephone,,"[0.5, +inf)",2.84,-0.0665
77,foreign_worker,,"[-inf, 0.5)",121.34,-1.7177


## 5) WOE/LR 系数检查

In [8]:
coef = scorecard_desc.coef_
coef_series = pd.Series(coef, index=X_train_woe.columns)
negative_coef_features = coef_series[coef_series < 0].sort_values()
coef_positive_ratio = float((coef > 0).sum() / len(coef))

print('[WOE/LR sanity]')
print(f'coef count        : {len(coef)}')
print(f'coef > 0 ratio    : {coef_positive_ratio:.4f}')
print(f'all coef > 0      : {bool(np.all(coef > 0))}')
if len(negative_coef_features) > 0:
    print('negative coef feats:')
    print(negative_coef_features.to_string())

[WOE/LR sanity]
coef count        : 20
coef > 0 ratio    : 0.9000
all coef > 0      : False
negative coef feats:
job                       -0.464671
present_residence_since   -0.216992


## 6) score_odds_reference 基准分检查

In [9]:
ref = scorecard_desc.score_odds_reference
ref_row = ref.iloc[(ref['评分'] - 750).abs().argsort()[:1]]
ref_row[['评分', '理论逾期率(%)', '好客户:坏客户']]

,评分,理论逾期率(%),好客户:坏客户
50,750,2.7778%,35.0:1


In [10]:
scorecard_desc.score_odds_reference

,评分,理论Odds(坏好比),好客户:坏客户,理论逾期率,理论逾期率(%),对数Odds
0,1050,0.0009,1120.0:1,0.000892,0.0892%,-7.0211
1,1044,0.0010,1045.0:1,0.000956,0.0956%,-6.9518
2,1038,0.0010,975.0:1,0.001025,0.1025%,-6.8825
3,1032,0.0011,909.7:1,0.001098,0.1098%,-6.8131
4,1026,0.0012,848.8:1,0.001177,0.1177%,-6.7438
...,...,...,...,...,...,...
96,474,0.6929,1.4:1,0.409297,40.9297%,-0.3669
97,468,0.7426,1.3:1,0.426155,42.6155%,-0.2976
98,462,0.7959,1.3:1,0.443186,44.3186%,-0.2282
99,456,0.8531,1.2:1,0.460352,46.0352%,-0.1589


In [11]:
scorecard_asc.score_odds_reference

,评分,理论Odds(坏好比),好客户:坏客户,理论逾期率,理论逾期率(%),对数Odds
0,450,0.0009,1120.0:1,0.000892,0.0892%,-7.0211
1,456,0.0010,1045.0:1,0.000956,0.0956%,-6.9518
2,462,0.0010,975.0:1,0.001025,0.1025%,-6.8825
3,468,0.0011,909.7:1,0.001098,0.1098%,-6.8131
4,474,0.0012,848.8:1,0.001177,0.1177%,-6.7438
...,...,...,...,...,...,...
96,1026,0.6929,1.4:1,0.409297,40.9297%,-0.3669
97,1032,0.7426,1.3:1,0.426155,42.6155%,-0.2976
98,1038,0.7959,1.3:1,0.443186,44.3186%,-0.2282
99,1044,0.8531,1.2:1,0.460352,46.0352%,-0.1589


## 7) 使用 feature_bin_stats 查看信用分与欺诈分效果

分别对 `descending` 信用评分和 `ascending` 欺诈评分做分箱效果统计。

In [12]:
from hscredit.report import feature_bin_stats

score_effect_df = pd.DataFrame({
    'target': y_test.to_numpy(),
    '信用评分': score_desc,
    '欺诈评分': score_asc,
}, index=X_test.index)

score_effect_df.head()

,target,信用评分,欺诈评分
115,0,738.694924,761.305076
346,0,540.408293,959.591707
328,0,510.360438,989.639562
974,0,607.676821,892.323179
587,0,461.125034,1038.874966


In [13]:
credit_score_stats = feature_bin_stats(
    score_effect_df,
    '信用评分',
    target='target',
    method='quantile',
    max_n_bins=10,
    min_bin_size=0.05,
    margins=True,
    return_cols=['样本总数', '样本占比', '坏样本数', '坏样本率', 'LIFT值', '累积LIFT值', '分档KS值'],
)

display(credit_score_stats)

,指标名称,指标含义,分箱标签,样本总数,样本占比,坏样本数,坏样本率,LIFT值,累积LIFT值,分档KS值
0,信用评分,信用评分,"[-inf, 341.409)",15,0.05,12,0.800000,2.6667,2.6667,0.119048
1,信用评分,信用评分,"[341.409, 433.9644)",45,0.15,30,0.666667,2.2222,2.3333,0.380952
2,信用评分,信用评分,"[433.9644, 521.1919)",75,0.25,31,0.413333,1.3778,1.8025,0.515873
3,信用评分,信用评分,"[521.1919, 569.7367)",45,0.15,6,0.133333,0.4444,1.4630,0.396825
4,信用评分,信用评分,"[569.7367, 598.3352)",15,0.05,3,0.200000,0.6667,1.4017,0.373016
5,信用评分,信用评分,"[598.3352, 612.1821)",15,0.05,2,0.133333,0.4444,1.3333,0.333333
6,信用评分,信用评分,"[612.1821, 630.6291)",15,0.05,0,0.000000,0.0000,1.2444,0.261905
7,信用评分,信用评分,"[630.6291, 678.8088)",30,0.10,4,0.133333,0.4444,1.1503,0.182540
8,信用评分,信用评分,"[678.8088, 767.0153)",30,0.10,2,0.066667,0.2222,1.0526,0.071429
9,信用评分,信用评分,"[767.0153, +inf)",15,0.05,0,0.000000,0.0000,1.0000,0.000000


In [14]:
fraud_score_stats = feature_bin_stats(
    score_effect_df,
    '欺诈评分',
    target='target',
    method='quantile',
    max_n_bins=10,
    min_bin_size=0.05,
    margins=True,
    return_cols=['样本总数', '样本占比', '坏样本数', '坏样本率', 'LIFT值', '累积LIFT值', '分档KS值'],
)

display(fraud_score_stats)

,指标名称,指标含义,分箱标签,样本总数,样本占比,坏样本数,坏样本率,LIFT值,累积LIFT值,分档KS值
0,欺诈评分,欺诈评分,"[-inf, 732.9847)",15,0.05,0,0.000000,0.0000,0.0000,0.071429
1,欺诈评分,欺诈评分,"[732.9847, 821.1912)",30,0.10,2,0.066667,0.2222,0.1481,0.182540
2,欺诈评分,欺诈评分,"[821.1912, 869.3709)",30,0.10,4,0.133333,0.4444,0.2667,0.261905
3,欺诈评分,欺诈评分,"[869.3709, 887.8179)",15,0.05,0,0.000000,0.0000,0.2222,0.333333
4,欺诈评分,欺诈评分,"[887.8179, 901.6648)",15,0.05,2,0.133333,0.4444,0.2540,0.373016
5,欺诈评分,欺诈评分,"[901.6648, 930.2633)",15,0.05,3,0.200000,0.6667,0.3056,0.396825
6,欺诈评分,欺诈评分,"[930.2633, 978.8081)",45,0.15,6,0.133333,0.4444,0.3434,0.515873
7,欺诈评分,欺诈评分,"[978.8081, 1066.0356)",75,0.25,31,0.413333,1.3778,0.6667,0.380952
8,欺诈评分,欺诈评分,"[1066.0356, 1158.591)",45,0.15,30,0.666667,2.2222,0.9123,0.119048
9,欺诈评分,欺诈评分,"[1158.591, +inf)",15,0.05,12,0.800000,2.6667,1.0000,0.000000


In [ ]:
from pathlib import Path
import tempfile
import traceback

scorecard_export_root = Path(tempfile.mkdtemp(prefix='hscredit_scorecard_exports_'))
scorecard_export_root

## 8) scorecard 导出与离线一致性验证

依次验证以下导出路径：

1. `pickle` 离线模型文件
2. `json` 规则文件
3. Python 部署代码
4. PMML 文件

说明：
- `pickle` 期望与原模型完全一致。
- `json` 规则导出当前按规则分数字典回载，允许存在很小的精度损失。
- Python 部署代码使用导出的规则直接对原始样本打分，理论上应与原模型几乎一致。
- PMML 依赖 `java`、`sklearn-pandas`、`sklearn2pmml`、`pypmml`；若环境仍缺少任一依赖，会在结果里记录原因。

In [6]:
from pathlib import Path
import tempfile
import traceback

export_summary_rows = []
export_artifacts = {}
scorecard_export_root = Path(tempfile.mkdtemp(prefix='hscredit_scorecard_exports_'))

scorecard_models = {
    'descending': scorecard_desc,
}

if 'scorecard_asc' in globals():
    scorecard_models['ascending'] = scorecard_asc
else:
    scorecard_models['ascending'] = ScoreCard(
        pdo=60,
        rate=2,
        base_odds=35,
        base_score=750,
        direction='ascending',
        binner=binner,
    )
    scorecard_models['ascending'].fit(X_train_woe, y_train, input_type='woe')

for model_name, card in scorecard_models.items():
    model_dir = scorecard_export_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    reference_scores = card.predict(X_test, input_type='raw')
    model_results = {}

    pickle_path = model_dir / 'scorecard.pkl'
    card.save_pickle(str(pickle_path))
    pickle_card = ScoreCard.load_pickle(str(pickle_path))
    pickle_scores = pickle_card.predict(X_test, input_type='raw')
    model_results['pickle'] = {
        'path': str(pickle_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - pickle_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - pickle_scores))),
        'status': 'ok',
    }

    json_path = model_dir / 'scorecard.json'
    card.export(to_json=str(json_path), include_meta=True)
    json_card = ScoreCard(binner=binner)
    json_card.load(str(json_path))
    json_scores = json_card.predict(X_test, input_type='raw')
    model_results['json'] = {
        'path': str(json_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - json_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - json_scores))),
        'status': 'ok',
    }

    py_path = model_dir / 'scorecard_deploy.py'
    py_namespace = {}
    py_code = card.export_deployment_code(language='python', output_file=str(py_path), decimal=12)
    exec(py_code, py_namespace)
    py_scores = X_test.apply(lambda row: py_namespace['calculate_score'](row.to_dict()), axis=1).to_numpy()
    model_results['python'] = {
        'path': str(py_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - py_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - py_scores))),
        'status': 'ok',
    }

    pmml_path = model_dir / 'scorecard.pmml'
    try:
        from pypmml import Model

        card.export_pmml(str(pmml_path))
        pmml_model = Model.load(str(pmml_path))
        pmml_output = pmml_model.predict(X_test)
        pmml_score_col = 'score' if 'score' in pmml_output.columns else pmml_output.columns[-1]
        pmml_scores = pmml_output[pmml_score_col].to_numpy(dtype=float)
        model_results['pmml'] = {
            'path': str(pmml_path),
            'score_column': pmml_score_col,
            'max_abs_diff': float(np.max(np.abs(reference_scores - pmml_scores))),
            'mean_abs_diff': float(np.mean(np.abs(reference_scores - pmml_scores))),
            'status': 'ok',
        }
    except Exception as exc:
        model_results['pmml'] = {
            'path': str(pmml_path),
            'status': 'failed',
            'error_type': type(exc).__name__,
            'error': str(exc),
            'traceback_tail': traceback.format_exc().splitlines()[-5:],
        }

    export_artifacts[model_name] = model_results

    for export_type, result in model_results.items():
        export_summary_rows.append({
            'model': model_name,
            'export_type': export_type,
            'status': result['status'],
            'max_abs_diff': result.get('max_abs_diff'),
            'mean_abs_diff': result.get('mean_abs_diff'),
            'path': result['path'],
            'note': result.get('error', ''),
        })

export_summary_df = pd.DataFrame(export_summary_rows)
export_summary_df

模型已保存至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_qxwwuzs0\descending\scorecard.pkl
模型已保存至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_qxwwuzs0\ascending\scorecard.pkl


,model,export_type,status,max_abs_diff,mean_abs_diff,path,note
0,descending,pickle,ok,0.000000,0.000000,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
1,descending,json,ok,0.043494,0.016024,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
2,descending,python,ok,42.162295,1.405410,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
3,descending,pmml,failed,NaN,NaN,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,No module named 'pypmml'
4,ascending,pickle,ok,0.000000,0.000000,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
5,ascending,json,ok,0.043494,0.016024,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
6,ascending,python,ok,42.162295,1.405410,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
7,ascending,pmml,failed,NaN,NaN,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,No module named 'pypmml'


In [7]:
diagnostic_rows = []

for model_name, card in scorecard_models.items():
    py_namespace = {}
    exec(card.export_deployment_code(language='python', decimal=12), py_namespace)
    reference_scores = card.predict(X_test, input_type='raw')
    py_scores = X_test.apply(lambda row: py_namespace['calculate_score'](row.to_dict()), axis=1).to_numpy()
    diffs = np.abs(reference_scores - py_scores)
    max_idx = int(np.argmax(diffs))
    diagnostic_rows.append({
        'model': model_name,
        'row_index': int(X_test.index[max_idx]),
        'max_abs_diff': float(diffs[max_idx]),
        'reference_score': float(reference_scores[max_idx]),
        'python_score': float(py_scores[max_idx]),
    })

pd.DataFrame(diagnostic_rows)

,model,row_index,max_abs_diff,reference_score,python_score
0,descending,856,42.162295,772.348617,730.186322
1,ascending,431,42.162295,1135.507081,1177.669376


In [ ]:
debug_feature_rows = []

for model_name, row_index in [('descending', 856), ('ascending', 431)]:
    card = scorecard_models[model_name]
    row = X_test.loc[[row_index]]
    x_woe = binner.transform(row, metric='woe')[card.feature_names_]
    sub_scores = card._woe_to_score(x_woe, card.feature_names_)[0]

    for feature, ref_sub_score in zip(card.feature_names_, sub_scores):
        rule = card.rules_[feature]
        selected_score = None
        raw_value = row.iloc[0][feature]
        for label, score in zip(rule['bin_labels'], rule['scores']):
            cond = card._bin_label_to_python_condition('val', label)
            if eval(cond, {'np': np}, {'val': raw_value}):
                selected_score = float(score)
                break

        if selected_score is None or abs(selected_score - ref_sub_score) > 1e-9:
            debug_feature_rows.append({
                'model': model_name,
                'row_index': row_index,
                'feature': feature,
                'raw_value': raw_value,
                'reference_sub_score': float(ref_sub_score),
                'selected_score': selected_score,
                'labels': list(rule['bin_labels']),
            })
            break

pd.DataFrame(debug_feature_rows)

In [ ]:
for model_name, results in export_artifacts.items():
    print(f'[{model_name}]')
    for export_type, result in results.items():
        if result['status'] == 'ok':
            print(
                f"  - {export_type:<6} max_abs_diff={result['max_abs_diff']:.12f} "
                f"mean_abs_diff={result['mean_abs_diff']:.12f}"
            )
        else:
            print(f"  - {export_type:<6} failed: {result['error_type']} - {result['error']}")

In [ ]:
export_summary_df.sort_values(['model', 'export_type']).reset_index(drop=True)

In [ ]:
export_conclusion_df = (
    export_summary_df
    .assign(
        acceptable=lambda df: df['max_abs_diff'].fillna(np.inf) <= 0.05,
        interpretation=lambda df: np.where(
            df['status'] != 'ok',
            '导出失败，需要单独排查环境或兼容性问题',
            np.where(
                df['max_abs_diff'].fillna(np.inf) <= 1e-9,
                '与原始评分几乎完全一致',
                np.where(
                    df['max_abs_diff'].fillna(np.inf) <= 0.05,
                    '存在极小精度损失，可接受',
                    '差异偏大，需要继续排查',
                )
            )
        )
    )
    [['model', 'export_type', 'status', 'max_abs_diff', 'mean_abs_diff', 'acceptable', 'interpretation', 'note']]
)

export_conclusion_df.sort_values(['model', 'export_type']).reset_index(drop=True)